# NM Assessment - Interactive Equilibrium Exploration

This notebook demonstrates manual strain adjustment to find equilibrium for combined N-M loading.

**Pattern:** Similar to unbalanced model in rc_bending - user manually adjusts strains to explore equilibrium.

**Goal:** Find strain state (eps_top, eps_bot) where:
- N_actual ≈ N_Ed (force equilibrium)
- M_actual ≥ M_Ed (moment capacity)

**Workflow:**
1. Set design loads: N_Ed, M_Ed
2. Adjust eps_top (usually fixed at concrete ultimate: -3.5‰)
3. Adjust eps_bot iteratively
4. Check equilibrium errors: ΔN, ΔM
5. Visualize stress-strain profile

In [ ]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt

from bmcs_cross_section.cs_design.shapes import RectangularShape
from bmcs_cross_section.cs_design.reinforcement import ReinforcementLayout, ReinforcementLayer
from bmcs_cross_section.cs_design.cross_section import CrossSection
from bmcs_cross_section.matmod.ec2_concrete import EC2Concrete
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement
from bmcs_cross_section.nm_assess.nm_assessment import NMAssessment

## 1. Create Cross-Section

Standard 300×500 mm section with reinforcement.

In [ ]:
# Geometry: 300x500 mm rectangular section
shape = RectangularShape(b=300.0, h=500.0)

# Material: C30/37 concrete
concrete = EC2Concrete(f_ck=30.0)

# Reinforcement: Steel B500
steel = SteelReinforcement(f_sy=500.0)

# Bottom layer: 4Ø20 at z=50mm
layer_bottom = ReinforcementLayer(
    z=50.0,
    A_s=1256.6,  # 4×π×10² mm²
    material=steel
)

# Top layer: 2Ø16 at z=450mm
layer_top = ReinforcementLayer(
    z=450.0,
    A_s=402.1,   # 2×π×8² mm²
    material=steel
)

reinforcement = ReinforcementLayout(layers=[layer_bottom, layer_top])

# Assemble cross-section
cs = CrossSection(
    shape=shape,
    concrete=concrete,
    reinforcement=reinforcement
)

print(f"Cross-section: {cs.shape.b:.0f}×{cs.h_total:.0f} mm")
print(f"Concrete: C{cs.concrete.f_ck:.0f}")
print(f"Steel: B{steel.f_sy:.0f}")
print(f"Reinforcement: {len(cs.reinforcement.layers)} layers")

## 2. Create NM Assessment State

Initialize with design loads and trial strain state.

In [ ]:
# Design loads (imposed)
N_Ed = 0.0      # Pure bending (no axial force)
M_Ed = 200.0    # 200 kNm design moment

# Initial trial strain state
eps_top = -0.0035  # Concrete ultimate compressive strain (EC2)
eps_bot = 0.0025   # Initial guess for bottom strain

# Create assessment
nm = NMAssessment(
    cs=cs,
    eps_top=eps_top,
    eps_bot=eps_bot,
    N_Ed=N_Ed,
    M_Ed=M_Ed
)

print(f"\n📋 Initial State:")
print(f"  Design loads: N_Ed = {N_Ed:.1f} kN, M_Ed = {M_Ed:.1f} kNm")
print(f"  Trial strains: ε_top = {eps_top*1000:.2f}‰, ε_bot = {eps_bot*1000:.2f}‰")
print(f"  Curvature: κ = {nm.kappa*1000:.4f} 1/m")

## 3. Check Current State

Compute forces and check equilibrium errors.

In [ ]:
# Get forces for current state
F_c, F_s, N_actual, M_actual = nm.get_forces()

print(f"\n⚙️ Current State Forces:")
print(f"  F_concrete = {F_c/1000:.2f} kN")
print(f"  F_steel    = {F_s/1000:.2f} kN")
print(f"  N_actual   = {N_actual:.2f} kN")
print(f"  M_actual   = {M_actual:.2f} kNm")

print(f"\n📊 Equilibrium Check:")
print(f"  ΔN = N_actual - N_Ed = {nm.N_error:+.2f} kN")
print(f"  ΔM = M_actual - M_Ed = {nm.M_error:+.2f} kNm")
print(f"  Utilization η = M_Ed/M_actual = {nm.utilization:.3f}")

print(f"\n✓ Status:")
if nm.is_equilibrium:
    print(f"  ✅ Equilibrium satisfied!")
    if nm.is_safe:
        print(f"  ✅ Cross-section is SAFE")
    else:
        print(f"  ⚠️ Cross-section is OVERSTRESSED")
else:
    print(f"  ⚠️ Equilibrium NOT satisfied")
    if abs(nm.N_error) > 1.0:
        print(f"     → Adjust ε_bot to reduce |ΔN|")
    if nm.M_error < 0:
        print(f"     → M_actual < M_Ed: insufficient capacity")

## 4. Visualize Stress-Strain Profile

Plot current state to understand the strain and stress distributions.

In [ ]:
# Use profile's built-in plotting method - NO CODE REDUNDANCY!
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

# NMAssessment.profile already provides the configured StressStrainProfile
nm.profile.plot_stress_strain_profile(
    ax_strain=ax_strain,
    ax_stress=ax_stress,
    show_resultants=True
)

# Add comprehensive title
state_info = f'ε_top={nm.eps_top*1000:.2f}‰, ε_bot={nm.eps_bot*1000:.2f}‰, κ={nm.kappa*1000:.4f} 1/m'
loads_info = f'N_Ed={nm.N_Ed:.1f}kN, M_Ed={nm.M_Ed:.1f}kNm'
forces_info = f'N={nm.N_actual:.1f}kN, M={nm.M_actual:.1f}kNm'
errors_info = f'ΔN={nm.N_error:+.1f}kN, ΔM={nm.M_error:+.1f}kNm'

fig.suptitle(
    f'NM Assessment | {state_info}\n{loads_info} | {forces_info} | {errors_info}',
    fontsize=12, fontweight='bold', y=0.98
)

plt.tight_layout()
plt.show()

print("\n📌 Visual Check:")
print(f"  - Top strain should show {nm.eps_top*1000:.2f}‰")
print(f"  - Bottom strain should show {nm.eps_bot*1000:.2f}‰")
print(f"  - Curvature indicator should show κ = {nm.kappa*1000:.4f} 1/m")

## 5. Iterative Adjustment - Trial 1

Adjust eps_bot to try to achieve equilibrium.

In [ ]:
# Try different eps_bot value
nm.eps_bot = 0.0030  # Increase bottom tension

print(f"\n🔄 Trial 1: eps_bot = {nm.eps_bot*1000:.2f}‰")
print(f"  Curvature: κ = {nm.kappa*1000:.4f} 1/m")
print(f"  N_actual = {nm.N_actual:.2f} kN (ΔN = {nm.N_error:+.2f} kN)")
print(f"  M_actual = {nm.M_actual:.2f} kNm (ΔM = {nm.M_error:+.2f} kNm)")
print(f"  Equilibrium: {nm.is_equilibrium}")

In [ ]:
# Plot updated state - reuse profile's method
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

nm.profile.plot_stress_strain_profile(
    ax_strain=ax_strain,
    ax_stress=ax_stress,
    show_resultants=True
)

state_info = f'ε_top={nm.eps_top*1000:.2f}‰, ε_bot={nm.eps_bot*1000:.2f}‰, κ={nm.kappa*1000:.4f} 1/m'
errors_info = f'ΔN={nm.N_error:+.1f}kN, ΔM={nm.M_error:+.1f}kNm'
fig.suptitle(f'Trial 1 | {state_info} | {errors_info}', fontsize=12, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

## 6. Iterative Adjustment - Trial 2

Continue adjusting based on error feedback.

In [ ]:
# Adjust based on previous result
nm.eps_bot = 0.0020  # Decrease if N_error was too negative

print(f"\n🔄 Trial 2: eps_bot = {nm.eps_bot*1000:.2f}‰")
print(f"  N_actual = {nm.N_actual:.2f} kN (ΔN = {nm.N_error:+.2f} kN)")
print(f"  M_actual = {nm.M_actual:.2f} kNm (ΔM = {nm.M_error:+.2f} kNm)")
print(f"  Equilibrium: {nm.is_equilibrium}")

In [ ]:
# Plot trial 2 - reuse profile's method
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

nm.profile.plot_stress_strain_profile(
    ax_strain=ax_strain,
    ax_stress=ax_stress,
    show_resultants=True
)

state_info = f'ε_top={nm.eps_top*1000:.2f}‰, ε_bot={nm.eps_bot*1000:.2f}‰'
errors_info = f'ΔN={nm.N_error:+.1f}kN, ΔM={nm.M_error:+.1f}kNm'
fig.suptitle(f'Trial 2 | {state_info} | {errors_info}', fontsize=12, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

## 7. Test with Axial Compression

Try a case with compression load (N_Ed < 0).

In [ ]:
# Change to compression case
nm.N_Ed = -500.0  # 500 kN compression
nm.M_Ed = 150.0   # 150 kNm moment
nm.eps_bot = 0.0010  # Less tension at bottom

print(f"\n🏗️ Compression Case:")
print(f"  Design: N_Ed = {nm.N_Ed:.1f} kN, M_Ed = {nm.M_Ed:.1f} kNm")
print(f"  Strains: ε_top = {nm.eps_top*1000:.2f}‰, ε_bot = {nm.eps_bot*1000:.2f}‰")
print(f"  N_actual = {nm.N_actual:.2f} kN (ΔN = {nm.N_error:+.2f} kN)")
print(f"  M_actual = {nm.M_actual:.2f} kNm (ΔM = {nm.M_error:+.2f} kNm)")

In [ ]:
# Plot compression case - reuse profile's method
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

nm.profile.plot_stress_strain_profile(
    ax_strain=ax_strain,
    ax_stress=ax_stress,
    show_resultants=True
)

loads_info = f'N_Ed={nm.N_Ed:.0f}kN, M_Ed={nm.M_Ed:.0f}kNm'
state_info = f'ε_top={nm.eps_top*1000:.2f}‰, ε_bot={nm.eps_bot*1000:.2f}‰'
errors_info = f'ΔN={nm.N_error:+.1f}kN, ΔM={nm.M_error:+.1f}kNm'
fig.suptitle(f'Compression Case | {loads_info} | {state_info} | {errors_info}', 
            fontsize=12, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

## 8. Summary Table

Display complete assessment summary.

In [ ]:
# Get complete summary
summary = nm.summary()

print("\n" + "="*60)
print("NM ASSESSMENT SUMMARY")
print("="*60)

print(f"\n📐 Cross-Section:")
print(f"  Dimensions: {summary['b']:.0f} × {summary['h']:.0f} mm")
print(f"  Concrete: C{summary['f_ck']:.0f}")

print(f"\n📊 Design Loads (Imposed):")
print(f"  N_Ed = {summary['N_Ed']:.1f} kN")
print(f"  M_Ed = {summary['M_Ed']:.1f} kNm")

print(f"\n🔧 Strain State:")
print(f"  ε_top = {summary['eps_top']*1000:.4f}‰")
print(f"  ε_bot = {summary['eps_bot']*1000:.4f}‰")
print(f"  κ = {summary['kappa']*1000:.6f} 1/m")

print(f"\n⚙️ Actual Forces:")
print(f"  N_actual = {summary['N_actual']:.2f} kN")
print(f"  M_actual = {summary['M_actual']:.2f} kNm")

print(f"\n📈 Equilibrium Errors:")
print(f"  ΔN = {summary['N_error']:+.2f} kN")
print(f"  ΔM = {summary['M_error']:+.2f} kNm")
print(f"  η = {summary['utilization']:.3f}")

print(f"\n✓ Status:")
print(f"  Equilibrium: {'✅ YES' if summary['is_equilibrium'] else '❌ NO'}")
print(f"  Safe: {'✅ YES' if summary['is_safe'] else '❌ NO'}")

print("\n" + "="*60)

## Summary

**Key Workflow:**

1. **Set design loads** - N_Ed, M_Ed are imposed constraints
2. **Fix eps_top** - Typically at concrete ultimate strain (-3.5‰)
3. **Adjust eps_bot iteratively** - Try different values
4. **Check equilibrium errors:**
   - ΔN = N_actual - N_Ed (target: ≈ 0)
   - ΔM = M_actual - M_Ed (target: ≥ 0)
5. **Visualize** - Stress-strain profile shows current state

**Interpretation:**

- **|ΔN| < 1 kN** → Force equilibrium satisfied
- **ΔM ≥ 0** → Sufficient moment capacity (M_actual ≥ M_Ed)
- **η ≤ 1.0** → Safe utilization

**Adjustment Strategy:**

- If ΔN > 0 (too much tension force) → Decrease eps_bot
- If ΔN < 0 (too much compression force) → Increase eps_bot
- If ΔM < 0 (insufficient moment) → Need more curvature or different section